In [9]:
# !pip install langchain langchain-groq langchain-community langgraph python-dotenv langchain-mcp-adapters langchain-chroma chromadb pypdf

In [10]:
import os
from pathlib import Path
from dotenv import load_dotenv

workspace_root = Path.cwd().resolve()
for candidate in [workspace_root, workspace_root.parent]:
    env_path = candidate / ".env"
    if env_path.exists():
        load_dotenv(env_path)
        break
else:
    load_dotenv()


In [11]:
groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    from dotenv import dotenv_values
    groq_api_key = dotenv_values(".env").get("GROQ_API_KEY")

if groq_api_key:
    os.environ["GROQ_API_KEY"] = groq_api_key
else:
    raise RuntimeError("GROQ_API_KEY is not set. Add it to your environment or create a .env file in the project root.")

GROQ_MODEL = "groq:llama-3.1-8b-instant"

In [12]:
from langchain.chat_models import init_chat_model

model = init_chat_model(GROQ_MODEL)
response = model.invoke("Hi")
print(response)
print("Cinebot's Brain is connected")

content="It's nice to meet you. Is there something I can help you with or would you like to chat?" additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 36, 'total_tokens': 59, 'completion_time': 0.026630723, 'completion_tokens_details': None, 'prompt_time': 0.001666039, 'prompt_tokens_details': None, 'queue_time': 0.052916911, 'total_time': 0.028296762}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019fe009-06aa-7282-8adf-6dc28a6e1715-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 36, 'output_tokens': 23, 'total_tokens': 59}
Cinebot's Brain is connected


In [13]:
print("hello")

hello


In [14]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool

@tool
def save_trip_demo(user_id: str, destination: str) -> str:
    """Save a trip to the database. Irreversible without manual cleanup."""
    return f"Trip to {destination} saved for {user_id}."  # standing in for a real DB write

agent = create_agent(
    model=GROQ_MODEL,
    tools=[save_trip_demo],
    middleware=[
        SummarizationMiddleware(
            model=GROQ_MODEL,
            trigger=("tokens", 4000),
            keep=("messages", 10),
        )
    ],
)

In [15]:
# Tool defined in the previous cell; no need to redefine it here.

In [16]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

# This older example is intentionally left as a reference.
# It uses a Groq-compatible model string and should be run only after the setup cell above.
agent = create_agent(
    model=GROQ_MODEL,
    tools=[save_trip_demo],
    middleware=[
        SummarizationMiddleware(
            model=GROQ_MODEL,
            trigger=("tokens", 4000),
            keep=("messages", 10),
        )
    ],
)

Summarization is text-oriented context compression. It does not resize, downsample, or otherwise compress image/audio/video payloads. Recent messages retained by keep still include their original multimodal blocks, while older multimodal messages that are summarized are represented only by the generated text summary. For image-heavy applications, store media in a filesystem or object store and pass URLs or file references through message history.

In [17]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def your_read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def your_send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model=GROQ_MODEL,
    tools=[your_read_email_tool, your_send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "your_send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "your_read_email_tool": False,
            }
        ),
    ],
)

In [18]:
config = {'configurable':{"thread_id":"hitl"}}

result =agent.invoke({"messages": [("user", "Send an email to my manager on pranay953ai@gmail.com, asking for a leave")]}, config=config)

In [19]:
result

{'messages': [HumanMessage(content='Send an email to my manager on pranay953ai@gmail.com, asking for a leave', additional_kwargs={}, response_metadata={}, id='2e429d88-feeb-44f5-a7c7-d912f2cbb366'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'rre9b85h0', 'function': {'arguments': '{"body":"Hello Manager, I am requesting a leave for personal reasons. Please let me know if this is acceptable.","recipient":"pranay953ai@gmail.com","subject":"Leave Request"}', 'name': 'your_send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 313, 'total_tokens': 367, 'completion_time': 0.063826947, 'completion_tokens_details': None, 'prompt_time': 0.018860938, 'prompt_tokens_details': None, 'queue_time': 0.048155652, 'total_time': 0.082687885}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq

```
User
  │
  │ "Send an email asking for leave"
  ▼
LLM
  │
  │ decides to call
  ▼
your_send_email_tool
  │
  │ requires approval
  ▼
INTERRUPT
  │
  ├── approve → send email
  ├── edit    → modify tool arguments
  └── reject  → don't send
```

In [20]:
# --- Core LangChain ---
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.tools import tool as tool_rt, ToolRuntime

In [21]:
# --- LangGraph (checkpointing, resuming) ---
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

In [22]:
from langchain.agents.middleware import (
    SummarizationMiddleware,
    HumanInTheLoopMiddleware,
    ModelCallLimitMiddleware,
    ToolCallLimitMiddleware,
    ModelFallbackMiddleware,
    PIIMiddleware,
    TodoListMiddleware,
    LLMToolSelectorMiddleware,
    ToolRetryMiddleware,
    ModelRetryMiddleware,
    LLMToolEmulator,
    ContextEditingMiddleware,
    ClearToolUsesEdit,
)

# Please start from here

Defining Tools for my Cinebot - Dummy as of now but ofcourse will be real as we have seen in last class.

In [23]:
@tool
def check_showtimes(movie_title: str) -> str:
    """Check available showtimes for a movie at the cinema."""
    fake_showtimes = {
        "interstellar": "7:00 PM and 10:15 PM",
        "dune part two": "9:30 PM only",
        "oppenheimer": "Sold out for tonight",
    }
    return fake_showtimes.get(movie_title.lower(), "No showtimes found for that title.")



In [24]:

@tool
def book_seats(movie_title: str, seat_count: int) -> str:
    """Book seats for a movie. Irreversible once confirmed."""
    return f"Booked {seat_count} seat(s) for {movie_title}."


In [25]:
@tool
def cancel_booking(booking_id: str) -> str:
    """Cancel an existing booking. Irreversible."""
    return f"Booking {booking_id} cancelled."


In [26]:
@tool
def check_order_status(booking_id: str) -> str:
    """Check the status of an existing booking."""
    return f"Booking {booking_id}: confirmed, 2 seats, Interstellar, 7:00 PM."


In [27]:
@tool
def get_refund_policy() -> str:
    """Get the cinema's refund policy -- exact wording, not to be paraphrased."""
    return "Refunds available up to 2 hours before showtime. No refunds after that."

In [28]:

@tool
def lookup_seat_map(movie_title: str, seat_number: str) -> str:
    """Look up a specific seat -- fails if the seat number format is wrong."""
    if not seat_number or not seat_number[0].isalpha():
        raise ValueError(f"Malformed seat number '{seat_number}' -- expected a letter+number like 'A12'.")
    return f"Seat {seat_number} for {movie_title}: available."


In [29]:
cinebot_tools = [check_showtimes, book_seats, cancel_booking, check_order_status, get_refund_policy, lookup_seat_map]


In [30]:
def pretty_print_messages(result):
    for i, message in enumerate(result.get("messages", []), 1):
        print(f"\n{'=' * 80}")
        print(f"Message {i}: {message.__class__.__name__}")
        print("=" * 80)

        # Basic message information
        print(f"ID: {getattr(message, 'id', None)}")

        # Message content
        content = getattr(message, "content", "")
        if content:
            print("\nContent:")
            print(content)

        # Tool calls
        tool_calls = getattr(message, "tool_calls", None)
        if tool_calls:
            print("\nTool Calls:")
            for tool in tool_calls:
                print(f"  • {tool['name']}")
                print(f"    Args: {tool['args']}")
                print(f"    ID:   {tool['id']}")

        # Tool message information
        tool_call_id = getattr(message, "tool_call_id", None)
        if tool_call_id:
            print(f"\nTool Call ID: {tool_call_id}")

        # Summarization information
        additional_kwargs = getattr(message, "additional_kwargs", {})
        if additional_kwargs.get("lc_source"):
            print(f"\nSource: {additional_kwargs['lc_source']}")

    print(f"\n{'=' * 80}")
    print("END OF MESSAGE HISTORY")
    print("=" * 80)

# Summarization Middleware

In [31]:
# plz add comment for token if > this summarise previous messages and keep last 3 messages only . help us to memorise it

In [32]:
summarizing_agent = create_agent(
    model=GROQ_MODEL,
    tools=cinebot_tools,
    middleware=[
        SummarizationMiddleware(
            model=GROQ_MODEL,
            trigger=("tokens", 300),
            keep=("messages", 1),
        )
    ],
)


In [33]:
result = summarizing_agent.invoke({"messages": [("user", "Is Interstellar showing tonight? also please make sure that you book me a ticket, refund me if it is not available,also share the refund policy for me to go through, also check my order status for book_1234")]})

In [34]:
pretty_print_messages(result)


Message 1: HumanMessage
ID: 661d9a30-1d9f-4f15-8352-232b83e9ebb0

Content:
Here is a summary of the conversation to date:

## SESSION INTENT

- The user's primary goal is to check if the movie "Interstellar" is showing tonight.
- The user also wants to ensure that a ticket is booked for them, and they want a refund if it's not available.
- The user requests to review the refund policy and check the status of their previous order (book_1234).

## SUMMARY

- The conversation began with the user inquiring about the movie "Interstellar" showing tonight.
- The user requested a ticket to be booked, with the condition of a refund if the movie is not available.
- The user also asked to review the refund policy and check the status of their previous order (book_1234).

## ARTIFACTS

None

## NEXT STEPS

- Verify the movie "Interstellar" showtime for tonight.
- Check availability and book a ticket for the user.
- Provide the refund policy to the user.
- Check the status of the user's previous o

# HITL (Human in the Loop)

In [35]:
guarded_agent = create_agent(
    model=GROQ_MODEL,
    tools=cinebot_tools,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"cancel_booking": {"allowed_decisions": ["approve", "edit", "reject", "respond"]}}
        ),
    ],
    checkpointer=InMemorySaver(),  # REQUIRED -- HITL needs to pause and later resume
)

config = {'configurable':{'thread_id':'hitl-demo-live'}}

In [36]:
result = guarded_agent.invoke({"messages": [("user", "Please cancel booking BK1042")]}, config=config)

In [37]:
print(result)

{'messages': [HumanMessage(content='Please cancel booking BK1042', additional_kwargs={}, response_metadata={}, id='6789c4a9-4c7a-4761-a140-f54a297686f9'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'q77n076rg', 'function': {'arguments': '{"booking_id":"BK1042"}', 'name': 'cancel_booking'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 566, 'total_tokens': 584, 'completion_time': 0.024176303, 'completion_tokens_details': None, 'prompt_time': 0.023042488, 'prompt_tokens_details': {'cached_tokens': 512}, 'queue_time': 0.044448951, 'total_time': 0.047218791}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fe009-14e7-7732-8e5d-0c5b8f0b3498-0', tool_calls=[{'name': 'cancel_booking', 'args': {'booking_id': 'BK1042'}, 'id': 'q77n076rg', 'type': 'tool_call'}], invalid_tool_c

In [38]:
state = guarded_agent.get_state(config)

In [39]:
state 

StateSnapshot(values={'messages': [HumanMessage(content='Please cancel booking BK1042', additional_kwargs={}, response_metadata={}, id='6789c4a9-4c7a-4761-a140-f54a297686f9'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'q77n076rg', 'function': {'arguments': '{"booking_id":"BK1042"}', 'name': 'cancel_booking'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 566, 'total_tokens': 584, 'completion_time': 0.024176303, 'completion_tokens_details': None, 'prompt_time': 0.023042488, 'prompt_tokens_details': {'cached_tokens': 512}, 'queue_time': 0.044448951, 'total_time': 0.047218791}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fe009-14e7-7732-8e5d-0c5b8f0b3498-0', tool_calls=[{'name': 'cancel_booking', 'args': {'booking_id': 'BK1042'}, 'id': 'q77n076rg', 'type': 'tool_ca

In [40]:
print(state.tasks[0])

PregelTask(id='476be0d7-59d1-63fa-1d0d-81f71c0d19a9', name='HumanInTheLoopMiddleware.after_model', path=('__pregel_pull', 'HumanInTheLoopMiddleware.after_model'), error=None, interrupts=(Interrupt(value={'action_requests': [{'name': 'cancel_booking', 'args': {'booking_id': 'BK1042'}, 'description': "Tool execution requires approval\n\nTool: cancel_booking\nArgs: {'booking_id': 'BK1042'}"}], 'review_configs': [{'action_name': 'cancel_booking', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='48f20e758f948745fba26f517ddc08ec'),), state=None, result=None)


In [41]:
resumed_result = guarded_agent.invoke(Command(resume={"decisions":[{"type":"approve"}]}),config=config)

In [42]:
print(resumed_result)

{'messages': [HumanMessage(content='Please cancel booking BK1042', additional_kwargs={}, response_metadata={}, id='6789c4a9-4c7a-4761-a140-f54a297686f9'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'q77n076rg', 'function': {'arguments': '{"booking_id":"BK1042"}', 'name': 'cancel_booking'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 566, 'total_tokens': 584, 'completion_time': 0.024176303, 'completion_tokens_details': None, 'prompt_time': 0.023042488, 'prompt_tokens_details': {'cached_tokens': 512}, 'queue_time': 0.044448951, 'total_time': 0.047218791}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fe009-14e7-7732-8e5d-0c5b8f0b3498-0', tool_calls=[{'name': 'cancel_booking', 'args': {'booking_id': 'BK1042'}, 'id': 'q77n076rg', 'type': 'tool_call'}], invalid_tool_c

In [43]:
def run_interactive_hitl_demo(agent, config):
    """A genuinely interactive HITL loop -- ask out loud, type the answer, watch it apply live."""
    state = agent.get_state(config)
    if not state.next:
        print("Nothing is currently paused for approval.")
        return

    print("The agent wants to call a guarded tool. Choose a decision:")
    print("  1) approve  -- run it exactly as proposed")
    print("  2) edit     -- run it, but change the booking_id first")
    print("  3) reject   -- block it, with a reason sent back to the agent")
    print("  4) respond  -- answer a question instead of deciding on the action")

    choice = input("Type 1, 2, 3, or 4: ").strip()

    if choice == "1":
        decision = {"type": "approve"}
    elif choice == "2":
        new_id = input("New booking_id to use instead: ").strip()
        decision = {"type": "edit", "args": {"booking_id": new_id}}
    elif choice == "3":
        reason = input("Reason for rejecting: ").strip()
        decision = {"type": "reject", "message": reason}
    elif choice == "4":
        answer = input("Your response to the agent: ").strip()
        decision = {"type": "respond", "message": answer}
    else:
        print("Not a valid choice -- try again.")
        return

    resumed = agent.invoke(Command(resume={"decisions": [decision]}), config=config)
    print()
    print("Agent's final response:", resumed["messages"][-1].content)


In [44]:
# Use the same Groq-backed model for all agent examples in this notebook
GROQ_MODEL = "groq:llama-3.1-8b-instant"

In [45]:
guarded_agent = create_agent(
    model=GROQ_MODEL,
    tools=cinebot_tools,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"cancel_booking": {"allowed_decisions": ["approve", "edit", "reject", "respond"]}}
        ),
    ],
    checkpointer=InMemorySaver(),  # REQUIRED -- HITL needs to pause and later resume
)



In [46]:
result = guarded_agent.invoke({"messages": [("user", "Please cancel booking BK1042")]}, config=config)

In [47]:
run_interactive_hitl_demo(guarded_agent, config)

The agent wants to call a guarded tool. Choose a decision:
  1) approve  -- run it exactly as proposed
  2) edit     -- run it, but change the booking_id first
  3) reject   -- block it, with a reason sent back to the agent
  4) respond  -- answer a question instead of deciding on the action
Not a valid choice -- try again.


> ⚙️ **Code Walkthrough:** `Command(resume={"decisions": [...]})` is how you hand a decision
> back to an agent that's paused mid-run. The `decisions` list has one entry per interrupted
> tool call (usually just one). Each decision is a dict with a `"type"` key matching one of the
> four options, plus whatever extra data that type needs — `edit` needs new `args`, `reject` and
> `respond` need a `message`, `approve` needs nothing else.

# Model Call Limit

In [48]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver

# Ensure the Groq model is available even if the setup cell was skipped.
if 'GROQ_MODEL' not in globals():
    GROQ_MODEL = "groq:llama-3.1-8b-instant"

call_limited_agent = create_agent(
    model=GROQ_MODEL,
    tools=cinebot_tools,
    checkpointer=InMemorySaver(),  # required for thread_limit to persist across calls
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=5,   # across the WHOLE conversation
            run_limit=2,      # per single .invoke() call
            exit_behavior="end",  # graceful stop, not an exception
        ),
    ],
)

In [49]:
result = call_limited_agent.invoke(
    {"messages": [("user", "Can you tell me cinema's refund policy? ")]},
    config={"configurable": {"thread_id": "call-limit-demo-4"}},
)

In [50]:
print(result)

{'messages': [HumanMessage(content="Can you tell me cinema's refund policy? ", additional_kwargs={}, response_metadata={}, id='b4ac49d1-8daa-4b2d-815a-90cc731e86fc'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'bqc41pgbk', 'function': {'arguments': '{}', 'name': 'get_refund_policy'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 570, 'total_tokens': 580, 'completion_time': 0.009980898, 'completion_tokens_details': None, 'prompt_time': 0.053936638, 'prompt_tokens_details': None, 'queue_time': 0.054336731, 'total_time': 0.063917536}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fe00a-0f0d-77a1-88bb-3bdf630eea5f-0', tool_calls=[{'name': 'get_refund_policy', 'args': {}, 'id': 'bqc41pgbk', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 57

In [51]:
result_invoke_2= call_limited_agent.invoke(
    {"messages": [("user", "cancel my booking B123? ")]},
    config={"configurable": {"thread_id": "call-limit-demo-4"}},
)

In [54]:
print(result_invoke_2)

{'messages': [HumanMessage(content="Can you tell me cinema's refund policy? ", additional_kwargs={}, response_metadata={}, id='b4ac49d1-8daa-4b2d-815a-90cc731e86fc'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'bqc41pgbk', 'function': {'arguments': '{}', 'name': 'get_refund_policy'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 570, 'total_tokens': 580, 'completion_time': 0.009980898, 'completion_tokens_details': None, 'prompt_time': 0.053936638, 'prompt_tokens_details': None, 'queue_time': 0.054336731, 'total_time': 0.063917536}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fe00a-0f0d-77a1-88bb-3bdf630eea5f-0', tool_calls=[{'name': 'get_refund_policy', 'args': {}, 'id': 'bqc41pgbk', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 57

In [55]:
result_invoke4 = call_limited_agent.invoke(
    {"messages": [("user", "Summarize my chat? ")]},
    config={"configurable": {"thread_id": "call-limit-demo-4"}},
)

In [56]:
print(result_invoke4)

{'messages': [HumanMessage(content="Can you tell me cinema's refund policy? ", additional_kwargs={}, response_metadata={}, id='b4ac49d1-8daa-4b2d-815a-90cc731e86fc'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'bqc41pgbk', 'function': {'arguments': '{}', 'name': 'get_refund_policy'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 570, 'total_tokens': 580, 'completion_time': 0.009980898, 'completion_tokens_details': None, 'prompt_time': 0.053936638, 'prompt_tokens_details': None, 'queue_time': 0.054336731, 'total_time': 0.063917536}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fe00a-0f0d-77a1-88bb-3bdf630eea5f-0', tool_calls=[{'name': 'get_refund_policy', 'args': {}, 'id': 'bqc41pgbk', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 57

# Model Fallback

In [57]:
resilient_agent = create_agent(
    model=GROQ_MODEL,     # primary, Groq instant model
    tools=cinebot_tools,
)

In [58]:
result = resilient_agent.invoke( {"messages": [("user", "Summarize my chat? ")]},)

In [59]:
resilient_agent = create_agent(
    model=GROQ_MODEL,     # primary, Groq instant model
    tools=cinebot_tools,
    middleware=[
        ModelFallbackMiddleware(
            GROQ_MODEL,   # fallback -- same Groq model, still works without extra setup
            # "ollama:llama3.2",   # a further, fully-local last resort -- uncomment if you have
                                    # `pip install langchain-ollama` AND a local Ollama server running.
                                    # Left commented here so this cell runs with nothing beyond
                                    # what Setup already installed.
        ),
    ],
)
print("Fallback chain: Groq primary -> Groq fallback.")
print("If the primary model call fails for any reason, this silently tries the next one.")

Fallback chain: Groq primary -> Groq fallback.
If the primary model call fails for any reason, this silently tries the next one.


In [60]:
result = resilient_agent.invoke( {"messages": [("user", "Summarize my chat? ")]},)

In [61]:
print(result)

{'messages': [HumanMessage(content='Summarize my chat? ', additional_kwargs={}, response_metadata={}, id='2cd172b3-1962-461a-826c-d0efba3a7ea0'), AIMessage(content="You've been provided with a list of available functions related to managing movie bookings and showtimes at a cinema, along with their descriptions and parameters.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 567, 'total_tokens': 597, 'completion_time': 0.05614617, 'completion_tokens_details': None, 'prompt_time': 0.031824318, 'prompt_tokens_details': None, 'queue_time': 0.159675762, 'total_time': 0.087970488}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fe00a-cada-7da1-ad50-be7530bf2579-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 567, 'output_tokens': 30, 'total_tokens': 597})]}


# Tool Call Limit

In [62]:
tool_limited_agent = create_agent(
    model=GROQ_MODEL,
    tools=cinebot_tools,
    checkpointer=InMemorySaver(),
    middleware=[
        ToolCallLimitMiddleware(run_limit=8),                              # global, this turn
        ToolCallLimitMiddleware(tool_name="cancel_booking", thread_limit=2),  # tighter, one tool, whole conversation
    ],
)